# Exploração da base e seleção das variáveis

Este notebook caracteriza a base recém-montada e define quais variáveis seguem
para a etapa seguinte. Dele resultam a **Tabela 2** (estatísticas descritivas e
percentual de zeros) e a **Figura 2** (matriz de correlação) do artigo.

**Feature** designa a variável empregada na comparação entre municípios. Nem
toda coluna da base é feature: há colunas de identificação (nome, código do
IBGE) e colunas de contexto, mantidas apenas para consulta. O papel de cada
uma é definido no `dicionario_base.csv`.

**Recorte de 644 municípios.** A análise considera somente os municípios com
nota do IEGM (Índice de Efetividade da Gestão Municipal), calculado pelo
TCE-SP (Tribunal de Contas do Estado de São Paulo). A capital é fiscalizada
pelo TCM-SP (Tribunal de Contas do Município de São Paulo) e, por isso, não
recebe IEGM: permanece na base, mas fora desta análise e da modelagem.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Os notebooks ficam em notebooks/ e o código do projeto em src/. Estas duas
# linhas apontam o Python para src/, para os "from config import ..." abaixo
# funcionarem tanto rodando daqui quanto da raiz do projeto.
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ / "src"))

from config import (BASE_FINAL_CSV, DICIONARIO_CSV, DATA_PROCESSED,
                    POPULACAO_MINIMA, LIMIAR_REDUNDANCIA)
from estilo import aplicar_estilo, salvar
from figuras import (plot_hist_populacao, plot_boxplot_taxas,
                     plot_distribuicoes_log1p, plot_spearman)

aplicar_estilo()   # mesma fonte, cores e grade em todas as figuras do artigo
# Se algum import acima falhar, quase sempre é o editor apontando para outra
# instalação do Python. O caminho impresso aqui é o que precisa ter as
# bibliotecas do requirements.txt.
print("Python:", sys.executable)

# O código do IBGE é identificador, não quantidade: lido como texto para não
# virar número em nenhuma etapa.
base = pd.read_csv(BASE_FINAL_CSV, dtype={"codigo_ibge": str})
dic = pd.read_csv(DICIONARIO_CSV)
print(f"base_final: {base.shape[0]} municípios x {base.shape[1]} colunas")

A lista de colunas é lida do `dicionario_base.csv`, gerado pelo
`merge_bases.py` junto com a base, de modo a evitar a repetição manual dos
nomes. Para cada coluna o dicionário registra o **bloco** de origem
(criminalidade, socioeconômico ou gestão) e o **papel** desempenhado:

- `feature` - entra na comparação entre municípios;
- `feature_tier2` - medida válida, porém frágil para comparação: ou concentra
  quase todos os municípios no mesmo valor, ou só existe onde houve
  ocorrência. Permanece na base, fora da modelagem;
- demais papéis (`chave`, `rotulo`, `contexto`, `metadado`) - mantidos para
  consulta e para as tabelas do artigo, sem uso em cálculo.

Ler do dicionário faz com que qualquer mudança de papel no `merge_bases.py` se
propague automaticamente até aqui.

Um efeito imediato: a análise trabalha com **9** taxas criminais, e não com as
10 listadas no `relatorio_base.txt`. O roubo de carga é `feature_tier2`, pois
57% dos municípios não registraram nenhum caso em três anos.

In [ ]:
# bloco_de["taxa_cvli"] devolve "criminalidade". Este atalho é usado o
# notebook inteiro para saber de que fonte cada coluna veio.
bloco_de = dic.set_index("coluna")["bloco"]
ORDEM_BLOCOS = ["criminalidade", "socioeconomico", "gestao"]
# Ordena as features por bloco (e por nome dentro do bloco), para as tabelas
# e a Figura 2 saírem sempre agrupadas na mesma ordem.
features = sorted(dic.loc[dic["papel"] == "feature", "coluna"],
                  key=lambda c: (ORDEM_BLOCOS.index(bloco_de[c]), c))
# As taxas criminais são reconhecidas pelo bloco, e não pelo nome começar com
# "taxa_": taxa_alfabetizacao e taxa_urbanizacao começam igual e são
# socioeconômicas.
taxas = [c for c in features if bloco_de[c] == "criminalidade"]
ordinais = [c for c in features if bloco_de[c] == "gestao"]

# flag_sem_iegm marca a capital; o "~" inverte a marcação e sobram os 644.
modelagem = base[~base["flag_sem_iegm"]].copy()
print(f"{len(features)} features em {len(modelagem)} municípios:")
print(pd.Series([bloco_de[c] for c in features]).value_counts().reindex(ORDEM_BLOCOS).to_string())

## 0. Caracterização inicial

Levantamento do conteúdo da base, dos valores ausentes e da dispersão das
variáveis. Nenhuma decisão de modelagem é tomada nesta seção: ela delimita o
material de trabalho e formula as questões que as seções seguintes respondem.

In [ ]:
# Quantas linhas, quais colunas, de que tipo e quantas estão preenchidas.
base.info()

In [ ]:
# Só as colunas que têm algum valor faltando.
ausentes = base.isna().sum()
ausentes[ausentes > 0]

Os ausentes são de dois tipos e nenhum deles decorre de falha de
coleta.

As colunas do IEGM faltam apenas na capital, pelo motivo já exposto. As duas
proporções de veículo faltam nos municípios sem nenhum roubo ou furto de
veículo em três anos, caso em que o denominador da proporção é zero.

A consequência para a etapa seguinte é direta: como a ausência não é
aleatória, a imputação por média ou mediana não se justifica. Os municípios
sem IEGM saem da modelagem e as duas proporções de veículo não são usadas como
feature.

In [ ]:
# describe() resume cada coluna: contagem, média, desvio padrão, mínimo,
# quartis e máximo. O .T vira a tabela de lado (uma linha por variável), que
# fica bem mais fácil de ler quando são muitas colunas.
for b in ORDEM_BLOCOS:
    cols = [c for c in features if bloco_de[c] == b]
    print(f"\n--- {b} ---")
    display(modelagem[cols].describe().round(2).T)

Três aspectos das tabelas merecem destaque.

**Nas taxas criminais, a média supera sistematicamente a mediana** e o máximo
fica muito acima do terceiro quartil, valor que separa os 75% menores do
restante. É o padrão de distribuições com *cauda longa à direita*: a maioria
dos municípios concentrada em valores baixos e poucos casos extremos que
elevam a média. O ponto retorna na seção 2, por afetar o cálculo de distância
entre municípios.

**No bloco socioeconômico, as duas variáveis monetárias medem objetos
distintos.** O PIB per capita alcança R$ 580 mil por habitante, enquanto a
renda domiciliar mediana não ultrapassa R$ 2.500. Não há contradição: o PIB
per capita divide toda a produção municipal pelo número de moradores, de modo
que uma refinaria ou usina eleva o indicador sem contrapartida na renda local.

**No IEGM, as notas concentram-se na parte baixa da escala.** A escala vai de
1 (conceito C) a 5 (conceito A) e o valor de cada município é a média dos
exercícios de 2022 a 2024, o que explica os valores fracionários. As medianas
das sete dimensões situam-se entre 1,0 e 2,7. Variáveis concentradas
discriminam pouco, ponto retomado na seção seguinte.

In [ ]:
# Escala log no eixo x: sem ela, São Paulo e as poucas cidades grandes
# esticam o gráfico e os 600 municípios pequenos viram uma barra só.
fig = plot_hist_populacao(base["populacao"], POPULACAO_MINIMA)
salvar(fig, "figura_eda_populacao")

Cerca de um quarto dos municípios paulistas (149 de 645) tem
menos de 5.000 habitantes, o que expõe uma limitação conhecida dos indicadores
em forma de taxa. Em um município de 3.000 habitantes, uma única ocorrência em
três anos resulta em taxa de 11 por 100 mil habitantes/ano: o valor é correto,
mas não é comparável aos 11 por 100 mil de um município de 300 mil habitantes,
já que o primeiro se altera substancialmente com um caso a mais.

A base sinaliza esses municípios com a coluna `flag_pop_pequena`. Eles não são
excluídos, pois descartar um quarto do estado alteraria a pergunta de
pesquisa; permanecem marcados para isolamento posterior, em análise de
sensibilidade.

In [ ]:
# Boxplot: a caixa cobre os 50% do meio, a linha dentro dela é a mediana e os
# pontos soltos são os municípios muito acima do resto. O eixo y está em
# escala log para as nove taxas, que têm ordens de grandeza diferentes,
# caberem no mesmo gráfico.
fig = plot_boxplot_taxas(modelagem, taxas)
salvar(fig, "figura_eda_boxplot_taxas")

In [ ]:
# nlargest(10, coluna) devolve os dez municípios com o maior valor na coluna.
def top10(coluna):
    return (modelagem.nlargest(10, coluna)[["municipio", "populacao", coluna]]
            .round(1).reset_index(drop=True))

# As duas listas lado a lado, para dar para comparar quem aparece em cada uma
# (repare na coluna de população).
pd.concat([top10("taxa_cvli"), top10("taxa_roubo_outros")], axis=1,
          keys=["taxa_cvli", "taxa_roubo_outros"])

As duas listas não têm municípios em comum, e a coluna de
população explica a diferença.

No topo do CVLI predominam municípios pequenos, com população mediana em torno
de 7 mil habitantes. Trata-se do efeito descrito acima: com poucos moradores,
dois ou três casos em três anos bastam para liderar o ranking. O valor é
verdadeiro, mas instável, pois a ausência de casos no período seguinte desloca
o mesmo município para o fim da lista.

No topo do roubo aparecem municípios médios e grandes, com população mediana
próxima de 317 mil habitantes, volume suficiente para estabilizar a taxa.

As duas taxas, portanto, não medem o mesmo fenômeno nem se comportam do mesmo
modo. É o primeiro argumento a favor de levar as taxas desagregadas para a
modelagem, em vez de compô-las em um índice único de criminalidade.

In [ ]:
# Três jeitos de ver se uma nota separa ou não os municípios: qual valor mais
# se repete, que fatia dos municípios está nele e quantos valores diferentes
# a coluna assume.
resumo_iegm = pd.DataFrame({
    "valor mais comum": modelagem[ordinais].mode().iloc[0],
    "% no valor mais comum": (modelagem[ordinais]
                              .apply(lambda s: s.value_counts(normalize=True).max()) * 100).round(0),
    "valores distintos": modelagem[ordinais].nunique(),
})
resumo_iegm

As notas do IEGM variam pouco. O caso extremo é o
`i_planejamento_ord`, em que 70% dos municípios receberam exatamente a mesma
nota. Uma variável concentrada dessa forma contribui pouco para diferenciar um
município do outro.

## 1. Tabela 2 - descritivas das taxas e percentual de zeros

**`% zeros`.** Zero em uma taxa criminal não indica baixa incidência, e sim
ausência de registro em três anos. Uma variável com 30% de zeros reúne na
mesma coluna dois tipos de município: os que apresentam o fenômeno em grau
baixo e os que não o apresentam. Nenhuma transformação matemática desfaz essa
mistura, de modo que resta dimensioná-la desde já.

**`assimetria`.** Mede a inclinação da distribuição. Zero indica simetria;
valores positivos indicam cauda longa à direita, caso de todas as taxas
analisadas. Quanto maior o valor, maior a concentração em municípios de baixa
taxa somada a poucos valores muito altos.

In [ ]:
tabela2 = pd.DataFrame({
    "média": modelagem[taxas].mean(),
    "mediana": modelagem[taxas].median(),
    "desvio": modelagem[taxas].std(),
    "máx": modelagem[taxas].max(),
    "% zeros": (modelagem[taxas] == 0).mean() * 100,
    # skew() é a assimetria: 0 = simétrica, positiva = cauda longa à direita.
    "assimetria": modelagem[taxas].skew(),
}).round(2)
# Fica salva em disco porque é uma tabela do artigo, não um resultado de
# passagem.
tabela2.to_csv(DATA_PROCESSED / "tabela2_descritivas.csv")
tabela2

## 2. As distribuições antes e depois do `log1p`

A cauda longa compromete qualquer cálculo baseado em distância entre
municípios, que é exatamente o que a clusterização utiliza. Com uma coluna
variando de 0 a 950 e outra de 0 a 100, a primeira determina quase sozinha a
semelhança entre municípios, não por relevância, mas por estar em escala
maior.

A solução usual é o logaritmo, que comprime os valores altos e preserva os
baixos, de maneira a trazer a cauda para junto do corpo da distribuição.
Adota-se o `log1p`, isto é, `log(1 + x)`: como o logaritmo de zero é
indefinido e as taxas contêm muitos zeros, somar 1 antes mantém o zero em
zero, sem imputar nem descartar dados.

A transformação é aplicada às 9 taxas criminais e às 2 variáveis monetárias
(PIB per capita e renda mediana), as de cauda longa. Percentuais, já limitados
a 0 e 100, e notas de 1 a 5 dispensam o tratamento.

In [ ]:
monetarias = ["pib_percapita", "renda_domiciliar_mediana"]
log_cols = taxas + monetarias

# Cada variável aparece duas vezes: à esquerda como está na base, à direita
# depois do log1p, com a assimetria no título de cada uma.
fig = plot_distribuicoes_log1p(modelagem, log_cols)
salvar(fig, "figura_distribuicoes_log1p")

Após o `log1p`, diversas taxas passam a apresentar assimetria
**negativa**, com a cauda voltada para a esquerda. Isso não indica excesso de
transformação.

A causa é a massa de zeros. O `log1p` mantém o zero em zero, enquanto os
demais valores se deslocam para a faixa de 3 a 4, o que cria um pico isolado à
esquerda, responsável pela assimetria negativa. A quantidade de zeros nem
precisa ser alta: o estupro tem 1% deles e ainda assim atinge −2,3, porque
esses poucos municípios ficam a quase quatro unidades do restante.

A tabela abaixo testa a explicação: se a causa forem os zeros, calcular a
assimetria apenas entre os municípios com valor positivo deve produzir valores
próximos de zero.

In [ ]:
pd.DataFrame({
    "% zeros": (modelagem[taxas] == 0).mean() * 100,
    "assimetria bruta": modelagem[taxas].skew(),
    "assimetria log1p": np.log1p(modelagem[taxas]).skew(),
    # A mesma conta da coluna anterior, mas só com quem tem valor > 0: é o
    # teste de que o negativo vem dos zeros, e não da transformação.
    "assimetria log1p sem zeros": [np.log1p(modelagem.loc[modelagem[c] > 0, c]).skew()
                                   for c in taxas],
}).round(2)

A hipótese se confirma: sem os zeros, a assimetria não ultrapassa
0,7 em módulo em nenhuma das nove taxas, contra os +1,1 a +3,9 anteriores a
qualquer transformação. O `log1p` cumpre sua função e fica mantido.

Os zeros permanecem como problema de outra natureza, que transformação alguma
resolve, por indicarem ausência do fenômeno e não erro de escala. Serão
tratados na análise de sensibilidade, com o isolamento dos municípios marcados
por `flag_pop_pequena`.

## 3. Figura 2 - a matriz de correlação

A correlação indica se duas variáveis variam conjuntamente: próxima de +1
quando crescem juntas, de −1 quando uma cresce e a outra decresce, e de 0 na
ausência de relação. Aqui serve para verificar se algum par de features mede o
mesmo atributo duas vezes.

Emprega-se a correlação de **Spearman**, e não a de Pearson. Pearson opera
sobre os valores e capta apenas relações lineares; Spearman substitui cada
valor por sua posição no ranking e capta qualquer relação monotônica. Dada a
cauda longa das taxas, trabalhar com posições é mais seguro: um município
atípico torna-se o primeiro colocado, em vez de distorcer o cálculo inteiro.

Na figura, as variáveis aparecem agrupadas por bloco, com linhas cinza
separando os três, o que permite a leitura por quadrantes: correlação alta
dentro de um bloco é esperada, entre blocos distintos é o que interessa.
Apenas os pares com |ρ| ≥ 0,5 recebem o valor anotado, de modo a preservar a
legibilidade da figura.

In [ ]:
# corr() devolve a matriz de todos contra todos: 22 x 22.
rho = modelagem[features].corr(method="spearman")

fig = plot_spearman(rho, bloco_de, ORDEM_BLOCOS)
salvar(fig, "figura2_spearman")

## 4. Poda por redundância

O critério é objetivo: features com |ρ| acima de `LIMIAR_REDUNDANCIA` (0,85,
definido no `config.py`) medem praticamente o mesmo atributo, e mantê-las
atribuiria peso duplicado a esse aspecto na comparação entre municípios, de
modo que uma delas seria removida.

O limiar é uma escolha deste trabalho, não uma regra universal, razão pela
qual fica no `config.py`, em local único e explícito, e não espalhado pelos
notebooks.

In [ ]:
# Percorre cada par de features uma vez só: o features[i + 1:] evita comparar
# a coluna com ela mesma e evita repetir o mesmo par ao contrário.
pares = [(a, b, round(rho.loc[a, b], 3))
         for i, a in enumerate(features) for b in features[i + 1:]
         if abs(rho.loc[a, b]) > LIMIAR_REDUNDANCIA]

if pares:
    display(pd.DataFrame(pares, columns=["a", "b", "rho"]))
else:
    print(f"Nenhum par com |rho| > {LIMIAR_REDUNDANCIA}: as {len(features)} "
          "features seguem para o pré-processamento.")

# Mesmo sem nenhum par passar do limiar, vale ver quais chegaram mais perto.
# A matriz é simétrica, então olhamos só o triângulo de baixo (tril), com
# k=-1 para tirar a diagonal, que é sempre 1. O stack() transforma a matriz
# numa lista de pares, que dá para ordenar.
maiores = rho.where(np.tril(np.ones_like(rho, dtype=bool), k=-1)).stack()
maiores.abs().sort_values(ascending=False).head(8).round(3)

## 5. Conclusões

1. **Nenhuma feature é descartada.** O par mais correlacionado da matriz é
   `roubo_outros × roubo_veiculo`, com ρ = 0,76, abaixo do limiar de 0,85. Em
   seguida vêm a urbanização com a coleta de lixo (0,74) e com o esgoto
   (0,67), coerentes por descreverem infraestrutura urbana e também abaixo do
   limiar. As 22 features seguem integralmente para o notebook 03.
2. **O quadrante criminalidade × gestão é praticamente vazio.** A maior
   correlação entre indicador criminal e nota de gestão é 0,37 e a mediana do
   quadrante é 0,10, ou seja, a nota de gestão de um município informa pouco
   sobre a sua criminalidade, e vice-versa. Esse resultado sustenta a proposta
   do trabalho: blocos redundantes não acrescentariam informação ao serem
   combinados.
3. O `log1p` nas taxas e nas variáveis monetárias fica confirmado, e os zeros
   permanecem registrados como limitação a ser tratada posteriormente.
4. Saídas geradas: `data/processed/tabela2_descritivas.csv` e
   `figuras/figura2_spearman.png`.